In [6]:
import sys
from pathlib import Path

# Find the project root (the folder that contains "shared")
root = Path.cwd().resolve()
while root != root.parent and not (root / "shared").exists():
    root = root.parent

if not (root / "shared").exists():
    raise RuntimeError("Could not find project root containing 'shared/'")

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

print("Project root:", root)
print("sys.path[0]:", sys.path[0])

from shared.paths import get_experiment_paths
from shared.seed import seed_everything
from shared.visualization import show_grid, save_grid

Project root: D:\Projects\GANs
sys.path[0]: D:\Projects\GANs


In [ ]:
EXP = "01_mlp_gan"
paths = get_experiment_paths(EXP)

seed_everything(42)

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [11]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_ds = datasets.MNIST(
    root=str(paths.datasets_dir / "MNIST"),
    train=True,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_ds,
    batch_size=128,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

100%|██████████| 9.91M/9.91M [00:04<00:00, 2.39MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 72.0kB/s]
100%|██████████| 1.65M/1.65M [00:02<00:00, 700kB/s] 
100%|██████████| 4.54k/4.54k [00:00<00:00, 4.55MB/s]


In [8]:
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
z_dim = 100

class Generator(nn.Module):
    def __init__(self, z_dim=100):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, 256),
            nn.ReLU(True),
            nn.Linear(256, 512),
            nn.ReLU(True),
            nn.Linear(512, 1024),
            nn.ReLU(True),
            nn.Linear(1024, 28*28),
            nn.Tanh()
        )

    def forward(self, z):
        x = self.net(z)
        return x.view(-1, 1, 28, 28)

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(28*28, 1024),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(1024, 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(256, 1)
        )

    def forward(self, x):
        x = x.view(x.size(0), -1)        # flatten
        return self.net(x).view(-1)      # (N,)

In [ ]:
criterion = nn.BCEWithLogitsLoss()

G = Generator(z_dim).to(device)
D = Discriminator().to(device)

g_opt = optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
d_opt = optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))

epochs = 20
fixed_z = torch.randn(64, z_dim, device=device)

for epoch in range(1, epochs + 1):
    G.train(); D.train()

    for real, _ in train_loader:
        real = real.to(device, non_blocking=True)
        b = real.size(0)

        # Train D
        z = torch.randn(b, z_dim, device=device)
        fake = G(z).detach()

        d_real = D(real)
        d_fake = D(fake)

        real_labels = torch.full((b,), 0.9, device=device)  # label smoothing
        fake_labels = torch.zeros(b, device=device)

        d_loss = (criterion(d_real, real_labels) + criterion(d_fake, fake_labels)) / 2

        d_opt.zero_grad(set_to_none=True)
        d_loss.backward()
        d_opt.step()

        # Train G
        z = torch.randn(b, z_dim, device=device)
        fake = G(z)
        d_fake_for_g = D(fake)

        g_loss = criterion(d_fake_for_g, torch.ones(b, device=device))

        g_opt.zero_grad(set_to_none=True)
        g_loss.backward()
        g_opt.step()

    G.eval()
    with torch.no_grad():
        samples = G(fixed_z)

    save_grid(samples, paths.samples_dir / f"epoch_{epoch:03d}.png", nrow=8)

    torch.save({
        "epoch": epoch,
        "G": G.state_dict(),
        "D": D.state_dict(),
        "g_opt": g_opt.state_dict(),
        "d_opt": d_opt.state_dict(),
        "z_dim": z_dim
    }, paths.models_dir / f"ckpt_epoch_{epoch:03d}.pth")

    print(f"[MLP GAN] Epoch {epoch}/{epochs} | d_loss={d_loss.item():.4f} | g_loss={g_loss.item():.4f}")

[MLP GAN] Epoch 1/20 | d_loss=0.2889 | g_loss=3.4765
[MLP GAN] Epoch 2/20 | d_loss=0.3099 | g_loss=3.3093
[MLP GAN] Epoch 3/20 | d_loss=0.1906 | g_loss=6.5784
[MLP GAN] Epoch 4/20 | d_loss=0.6137 | g_loss=1.0398
[MLP GAN] Epoch 5/20 | d_loss=0.4113 | g_loss=12.4479
[MLP GAN] Epoch 6/20 | d_loss=0.2786 | g_loss=7.3631
[MLP GAN] Epoch 7/20 | d_loss=0.2906 | g_loss=2.7762
[MLP GAN] Epoch 8/20 | d_loss=0.3134 | g_loss=4.1112
[MLP GAN] Epoch 9/20 | d_loss=0.3063 | g_loss=4.2871
[MLP GAN] Epoch 10/20 | d_loss=0.4040 | g_loss=4.9719
[MLP GAN] Epoch 11/20 | d_loss=0.4254 | g_loss=1.8598
[MLP GAN] Epoch 12/20 | d_loss=0.3951 | g_loss=2.7290
[MLP GAN] Epoch 13/20 | d_loss=0.3954 | g_loss=1.9071
[MLP GAN] Epoch 14/20 | d_loss=0.4943 | g_loss=2.1129
[MLP GAN] Epoch 15/20 | d_loss=0.4598 | g_loss=1.6377
[MLP GAN] Epoch 16/20 | d_loss=0.4673 | g_loss=2.4536
[MLP GAN] Epoch 17/20 | d_loss=0.4660 | g_loss=2.2449
[MLP GAN] Epoch 18/20 | d_loss=0.4793 | g_loss=2.3040
[MLP GAN] Epoch 19/20 | d_loss=0.511